# **World Happiness Dataset - Activity 2**

Import Necessary Libraries

In [2]:
!pip install pandas matplotlib seaborn

# Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import sqlite3

Load CSV FIle

In [3]:
from google.colab import files
uploaded = files.upload()

Saving world_happiness_dataset.csv to world_happiness_dataset.csv


**Verify Loaded Data**

In [4]:
df = pd.read_csv("world_happiness_dataset.csv")
print(df.head())

       Country  Happiness_Score  GDP_per_Capita  Social_Support  \
0       Norway             6.25            1.39            0.82   
1      Denmark             3.61            1.27            0.43   
2      Iceland             4.68            0.87            0.54   
3  Switzerland             4.46            0.67            0.57   
4      Finland             6.67            1.55            0.45   

   Healthy_Life_Expectancy  Freedom_to_Make_Choices  Generosity  \
0                     81.6                     0.69        0.01   
1                     66.9                     0.48        0.43   
2                     63.4                     0.71        0.41   
3                     68.8                     0.93        0.32   
4                     75.4                     0.58        0.16   

   Perceptions_of_Corruption  
0                       0.71  
1                       0.53  
2                       0.72  
3                       0.52  
4                       0.10  


**Load Dataframe into SQLite Database**

In [5]:
# ── Step 1: Create a SQLite database connection ────────────────
# This creates a database file called happiness.db
# If it already exists it just connects to it
conn = sqlite3.connect('happiness.db')

# ── Step 2: Push the DataFrame into SQLite as a table ─────────
df.to_sql(
    'happiness',       # table name inside the database
    conn,              # connection to use
    if_exists='replace',  # if table exists already — replace it
    index=False        # don't write the pandas index as a column
)

# ── Step 3: Verify — read back the table to confirm ───────────
verify = pd.read_sql("SELECT * FROM happiness LIMIT 5", conn)
print("Database created successfully!")
print(f"   Table name : happiness")
print(f"   Total rows : {pd.read_sql('SELECT COUNT(*) as count FROM happiness', conn)['count'][0]}")
print(f"\nFirst 5 rows:")
print(verify.to_string())

Database created successfully!
   Table name : happiness
   Total rows : 20

First 5 rows:
       Country  Happiness_Score  GDP_per_Capita  Social_Support  Healthy_Life_Expectancy  Freedom_to_Make_Choices  Generosity  Perceptions_of_Corruption
0       Norway             6.25            1.39            0.82                     81.6                     0.69        0.01                       0.71
1      Denmark             3.61            1.27            0.43                     66.9                     0.48        0.43                       0.53
2      Iceland             4.68            0.87            0.54                     63.4                     0.71        0.41                       0.72
3  Switzerland             4.46            0.67            0.57                     68.8                     0.93        0.32                       0.52
4      Finland             6.67            1.55            0.45                     75.4                     0.58        0.16                   

In [6]:
# Check GDP range so we can set category boundaries
print(df['GDP_per_Capita'].describe())

count    20.000000
mean      1.149000
std       0.304785
min       0.600000
25%       0.907500
50%       1.170000
75%       1.395000
max       1.570000
Name: GDP_per_Capita, dtype: float64


# **GDP Categories**

Creates GDP categories (Low, Medium, High), Calculates average happiness per category, Ranks countries within each category

In [7]:
query1 = """
SELECT
    Country,
    GDP_per_Capita,
    Happiness_Score,
    GDP_Category,

    -- AVG uses window WITHOUT ORDER BY → true group average
    ROUND(AVG(Happiness_Score) OVER (
        PARTITION BY GDP_Category
    ), 2) AS Avg_Happiness_Per_Category,

    -- RANK uses window WITH ORDER BY → correct ranking
    DENSE_RANK() OVER (
        PARTITION BY GDP_Category
        ORDER BY Happiness_Score DESC
    ) AS Rank_Within_Category

FROM (
    SELECT *,
        CASE
            WHEN GDP_per_Capita < 0.9075 THEN 'Low'
            WHEN GDP_per_Capita <= 1.395 THEN 'Medium'
            ELSE                              'High'
        END AS GDP_Category
    FROM happiness
)
ORDER BY GDP_Category, Rank_Within_Category
"""

result1 = pd.read_sql(query1, conn)
print("=" * 65)
print("QUERY 1 — GDP Categories + Avg Happiness + Ranking")
print("=" * 65)
print(result1.to_string())

QUERY 1 — GDP Categories + Avg Happiness + Ranking
           Country  GDP_per_Capita  Happiness_Score GDP_Category  Avg_Happiness_Per_Category  Rank_Within_Category
0           Brazil            1.45             6.98         High                        5.55                     1
1          Finland            1.55             6.67         High                        5.55                     2
2        Australia            1.43             5.31         High                        5.55                     3
3            India            1.41             4.45         High                        5.55                     4
4           France            1.57             4.36         High                        5.55                     5
5           Canada            0.60             7.34          Low                        5.35                     1
6      Netherlands            0.87             6.41          Low                        5.35                     2
7          Iceland           

# **Comparision on Corruption**

Splits countries into high vs low corruption perception, Computes multiple averages, Compares them using a subquery.

In [8]:
query2 = """
SELECT
    -- Step 1: Label each country as High or Low corruption
    CASE
        WHEN Perceptions_of_Corruption < 0.525 THEN 'High Corruption'
        ELSE                                        'Low Corruption'
    END AS Corruption_Level,

    -- Step 2: Count countries in each group
    COUNT(Country) AS Total_Countries,

    -- Step 3: Calculate multiple averages per group
    ROUND(AVG(Perceptions_of_Corruption),       2) AS Avg_Perceptions_of_Corruption,
    ROUND(AVG(Happiness_Score),       2) AS Avg_Happiness,
    ROUND(AVG(GDP_per_Capita),        2) AS Avg_GDP,
    ROUND(AVG(Social_Support),        2) AS Avg_Social_Support,
    ROUND(AVG(Healthy_Life_Expectancy),2) AS Avg_Life_Expectancy,
    ROUND(AVG(Freedom_to_Make_Choices),2) AS Avg_Freedom,
    ROUND(AVG(Generosity),2) AS Avg_Generosity,

    -- Step 4: Compare group average against low corruption avg
    -- using a scalar subquery
    ROUND((
        SELECT AVG(Happiness_Score)
        FROM happiness
        WHERE Perceptions_of_Corruption >= 0.525
    ), 2) AS Low_Corruption_Avg_Happiness,

    -- Step 5: Difference between group avg and low corruption avg
    ROUND(AVG(Happiness_Score) - (
        SELECT AVG(Happiness_Score)
        FROM happiness
        WHERE Perceptions_of_Corruption >= 0.525
    ), 2) AS Difference_From_Low_Corruption

FROM happiness

-- Step 6: Group results by corruption level
GROUP BY Corruption_Level
ORDER BY Avg_Happiness DESC
"""

result2 = pd.read_sql(query2, conn)
print("=" * 65)
print("QUERY 2 — High vs Low Corruption Comparison")
print("=" * 65)
print(result2.to_string())

QUERY 2 — High vs Low Corruption Comparison
  Corruption_Level  Total_Countries  Avg_Perceptions_of_Corruption  Avg_Happiness  Avg_GDP  Avg_Social_Support  Avg_Life_Expectancy  Avg_Freedom  Avg_Generosity  Low_Corruption_Avg_Happiness  Difference_From_Low_Corruption
0  High Corruption               10                           0.26           5.51      1.1                0.59                64.78         0.69            0.30                          4.83                            0.68
1   Low Corruption               10                           0.75           4.83      1.2                0.66                59.80         0.63            0.31                          4.83                            0.00
